# 纯 A 股 ETF 池重建实验 - 手动运行版

请在聚宽Research环境中运行此代码

In [ ]:
# 池子定义
OLD_POOL = {
    "沪深300ETF": "510300.XSHG",
    "中证500ETF": "510500.XSHG", 
    "创业板ETF": "159915.XSHE",
    "科创50ETF": "588000.XSHG",
    "中证1000ETF": "512100.XSHG",
    "纳指ETF": "513100.XSHG",
    "标普500ETF": "513500.XSHG",
    "黄金ETF": "518880.XSHG",
    "国债ETF": "511010.XSHG",
    "医疗ETF": "512170.XSHG",
    "消费ETF": "159928.XSHE",
    "新能源ETF": "516160.XSHG",
}

NEW_POOL = {
    "沪深300ETF": "510300.XSHG",
    "中证500ETF": "510500.XSHG",
    "创业板ETF": "159915.XSHE",
    "科创50ETF": "588000.XSHG",
    "中证1000ETF": "512100.XSHG",
    "医疗ETF": "512170.XSHG",
    "消费ETF": "159928.XSHE",
    "新能源ETF": "516160.XSHG",
    "半导体ETF": "512480.XSHG",
    "军工ETF": "512660.XSHG",
    "银行ETF": "512800.XSHG",
    "计算机ETF": "512720.XSHG",
}

print('✅ 池子定义完成')
print(f'旧池: {len(OLD_POOL)}只ETF')
print(f'新池: {len(NEW_POOL)}只ETF')

In [ ]:
# 回测参数
START = "2020-01-01"
END = "2024-12-31"
COST = 0.001  # 单边成本0.1%
MOM_WINDOW = 20  # 动量窗口
HOLD_DAYS = 10   # 持有周期
TOP_N = 3        # 持仓数量

print(f'回测区间: {START} ~ {END}')
print(f'策略参数: 动量{MOM_WINDOW}日 / 持有{HOLD_DAYS}日 / Top{TOP_N}')

In [ ]:
# 回测函数
def run_backtest(pool, pool_name):
    print(f'\n运行 {pool_name}...')
    
    dates = get_trade_days(START, END)[::HOLD_DAYS]
    if dates[-1] != get_trade_days(START, END)[-1]:
        dates = list(dates) + [get_trade_days(START, END)[-1]]
    
    codes = list(pool.values())
    rets = []
    prev_holdings = []
    
    for i, d in enumerate(dates[:-1]):
        d_str = str(d)
        next_d_str = str(dates[i + 1])
        
        try:
            # 计算动量
            prices = get_price(codes, end_date=d_str, count=MOM_WINDOW+1, fields=["close"], panel=False)
            pivot = prices.pivot(index="time", columns="code", values="close").dropna(axis=1)
            
            if len(pivot) < MOM_WINDOW:
                continue
            
            mom = pivot.iloc[-1] / pivot.iloc[0] - 1
            selected = mom.nlargest(TOP_N).index.tolist()
            
            # 计算收益
            p0 = get_price(selected, end_date=d_str, count=1, fields=["close"], panel=False)
            p1 = get_price(selected, end_date=next_d_str, count=1, fields=["close"], panel=False)
            
            if len(p0) == 0 or len(p1) == 0:
                continue
            
            p0 = p0.pivot(index="time", columns="code", values="close").iloc[-1]
            p1 = p1.pivot(index="time", columns="code", values="close").iloc[-1]
            
            gross_ret = ((p1 / p0) - 1).mean()
            turnover = len(set(selected) - set(prev_holdings)) / TOP_N
            net_ret = gross_ret - turnover * COST * 2
            
            rets.append(net_ret)
            prev_holdings = selected
            
        except Exception as e:
            continue
    
    if not rets:
        return None
    
    import pandas as pd
    import numpy as np
    
    s = pd.Series(rets)
    cum = (1 + s).cumprod()
    
    periods_per_year = 252 / HOLD_DAYS
    ann = cum.iloc[-1] ** (periods_per_year / len(s)) - 1
    dd = (cum / cum.cummax() - 1).min()
    sharpe = s.mean() / s.std() * np.sqrt(periods_per_year) if s.std() > 0 else 0
    win = (s > 0).mean()
    
    return {
        'name': pool_name,
        'ann': ann,
        'dd': dd,
        'sharpe': sharpe,
        'win': win,
        'rets': s,
        'cum': cum,
    }

print('✅ 回测函数定义完成')

In [ ]:
# 运行回测（可能需要几分钟）
old_res = run_backtest(OLD_POOL, "旧池(v1.0)")
new_res = run_backtest(NEW_POOL, "新池(v2.0)")

In [ ]:
# 显示结果
print("=" * 70)
print("【回测结果对比】")
print("=" * 70)

if old_res and new_res:
    print(f"\n{'指标':<20s} {'旧池':<15s} {'新池':<15s} {'差异':<15s}")
    print("-" * 65)
    print(f"{'年化收益':<20s} {old_res['ann']:<15.2%} {new_res['ann']:<15.2%} {new_res['ann']-old_res['ann']:<+15.2%}")
    print(f"{'最大回撤':<20s} {old_res['dd']:<15.2%} {new_res['dd']:<15.2%} {new_res['dd']-old_res['dd']:<+15.2%}")
    print(f"{'夏普比率':<20s} {old_res['sharpe']:<15.2f} {new_res['sharpe']:<15.2f} {new_res['sharpe']-old_res['sharpe']:<+15.2f}")
    print(f"{'胜率':<20s} {old_res['win']:<15.1%} {new_res['win']:<15.1%} {new_res['win']-old_res['win']:<+15.1%}")

    print("\n" + "=" * 70)
    print("结论:")
    if new_res['ann'] < old_res['ann']:
        print("❌ 新池年化收益低于旧池")
    if new_res['dd'] < old_res['dd']:
        print("✅ 新池回撤小于旧池")
    else:
        print("❌ 新池回撤大于旧池")
    print("=" * 70)

In [ ]:
# 绘制净值曲线对比
import matplotlib.pyplot as plt

if old_res and new_res:
    plt.figure(figsize=(12, 6))
    plt.plot(old_res['cum'].values, label=f"旧池 (ann={old_res['ann']:.1%})")
    plt.plot(new_res['cum'].values, label=f"新池 (ann={new_res['ann']:.1%})")
    plt.title("ETF池对比 - 净值曲线")
    plt.xlabel("调仓次数")
    plt.ylabel("净值")
    plt.legend()
    plt.grid(True)
    plt.show()